# Multi-Residue Topology Corner Cases

Tests for PR #430 validation with manually-built molecules having complex residue topologies:
- Residues with 2, 3, and 4 external bonds to other residues
- Virtual site with parent atoms spanning 3 residues
- Noncontiguous residues with identical metadata
- Molecule spanning multiple chains
- Stereochemical isomers

Molecules are built atom-by-atom using `Molecule.add_atom()` / `add_bond()` with residue metadata.

In [1]:
import copy
import io

import numpy as np
import openmm
import openmm.unit
from openff.toolkit import ForceField as OFFForceField, Molecule, Topology
from openff.toolkit.typing.engines.smirnoff.parameters import VirtualSiteType
from openff.units import Quantity as OFFQuantity
from openmm.app import ForceField, Modeller, NoCutoff

from openmmforcefields.generators import SMIRNOFFTemplateGenerator

EU = openmm.unit.kilocalories_per_mole
FU = openmm.unit.kilocalories_per_mole / openmm.unit.angstroms
FF = "openff_no_water-3.0.0-alpha0.offxml"

In [2]:
def compute_energy(system, positions):
    system = copy.deepcopy(system)
    for i, force in enumerate(system.getForces()):
        force.setForceGroup(i)
    integrator = openmm.VerletIntegrator(0.001)
    ctx = openmm.Context(system, integrator, openmm.Platform.getPlatformByName("Reference"))
    ctx.setPositions(positions)
    ctx.computeVirtualSites()
    ctx.applyConstraints(integrator.getConstraintTolerance())
    energy = ctx.getState(getEnergy=True).getPotentialEnergy()
    forces = ctx.getState(getForces=True).getForces(asNumpy=True)
    component_energies = {}
    for i in range(system.getNumForces()):
        name = system.getForce(i).__class__.__name__
        component_energies[name] = ctx.getState(getEnergy=True, groups=(1 << i)).getPotentialEnergy()
    del ctx, integrator
    return energy, forces, component_energies


def get_permutation(sys1, sys2):
    """Map particles from sys1 ordering to sys2 ordering (atoms by order, vsites by parents)."""
    atoms1 = [i for i in range(sys1.getNumParticles()) if not sys1.isVirtualSite(i)]
    atoms2 = [i for i in range(sys2.getNumParticles()) if not sys2.isVirtualSite(i)]
    sites1 = [i for i in range(sys1.getNumParticles()) if sys1.isVirtualSite(i)]
    sites2 = [i for i in range(sys2.getNumParticles()) if sys2.isVirtualSite(i)]
    assert len(atoms1) == len(atoms2), f"Atom count: {len(atoms1)} vs {len(atoms2)}"
    assert len(sites1) == len(sites2), f"Vsite count: {len(sites1)} vs {len(sites2)}"

    a2o1 = {p: i for i, p in enumerate(atoms1)}
    a2o2 = {p: i for i, p in enumerate(atoms2)}

    def parents(sys, vi, a2o):
        vs = sys.getVirtualSite(vi)
        return tuple(a2o[vs.getParticle(j)] for j in range(vs.getNumParticles()))

    s2_by_parents = {}
    for si in sites2:
        s2_by_parents.setdefault(parents(sys2, si, a2o2), []).append(si)

    perm = [-1] * sys1.getNumParticles()
    for a1, a2 in zip(atoms1, atoms2):
        perm[a1] = a2
    for si in sites1:
        key = parents(sys1, si, a2o1)
        perm[si] = s2_by_parents[key].pop(0)
    return perm


def compare(name, mol_or_mols, off_ff=None, preload=False, register_mols=None):
    """Run both pathways and compare energies.

    Parameters
    ----------
    name : str
        Test name for printing.
    mol_or_mols : Molecule or list[Molecule]
        Molecule(s) to test. If a list, all are placed in the topology.
    off_ff : OFFForceField, optional
        Force field to use. Defaults to the standard one.
    preload : bool
        If True, pre-load templates to work around OpenMM constraint bug.
    register_mols : list[Molecule], optional
        Molecules to register with the template generator. If None, uses
        deduplicated mols from the topology. Useful when stereoisomers
        would create ambiguous templates.
    """
    if off_ff is None:
        off_ff = OFFForceField(FF)

    mols = mol_or_mols if isinstance(mol_or_mols, list) else [mol_or_mols]

    # --- Interchange (reference) ---
    off_top = Topology.from_molecules(mols)
    ic_sys = off_ff.create_openmm_system(off_top)

    # --- OMMFFS ---
    if register_mols is not None:
        unique_mols = register_mols
    else:
        seen = {}
        unique_mols = []
        for m in mols:
            smi = m.to_smiles()
            if smi not in seen:
                seen[smi] = m
                unique_mols.append(m)

    ff_string = off_ff.to_string()
    gen = SMIRNOFFTemplateGenerator(molecules=unique_mols, forcefield=ff_string)
    omm_ff = ForceField()
    omm_ff.registerTemplateGenerator(gen.generator)

    if preload:
        for um in unique_mols:
            ffxml = gen.generate_residue_template(um)
            omm_ff.loadFile(io.StringIO(ffxml))

    omm_top = off_top.to_openmm()
    positions = off_top.get_positions().to_openmm()

    modeller = Modeller(omm_top, positions)
    modeller.addExtraParticles(omm_ff)

    of_sys = omm_ff.createSystem(modeller.topology, nonbondedMethod=NoCutoff)

    # --- Compare ---
    perm = get_permutation(of_sys, ic_sys)

    of_e, of_f, of_comp = compute_energy(of_sys, modeller.positions)

    inv_perm = [0] * len(perm)
    for i, j in enumerate(perm):
        inv_perm[j] = i
    ic_positions = [modeller.positions[inv_perm[j]] for j in range(len(perm))]
    ic_e, ic_f, ic_comp = compute_energy(ic_sys, ic_positions)

    ic_f_permuted = np.array([ic_f[perm[i]].value_in_unit(FU) for i in range(len(perm))]) * FU

    de = abs((of_e - ic_e).value_in_unit(EU))
    c_of = of_sys.getNumConstraints()
    c_ic = ic_sys.getNumConstraints()

    components = set(of_comp) & set(ic_comp)
    print(f"\n{'Component':<28} {'OMMFFS':>18} {'Interchange':>18} {'Delta':>14}")
    print("-" * 80)
    for key in sorted(components):
        t = of_comp[key].value_in_unit(EU)
        r = ic_comp[key].value_in_unit(EU)
        print(f"{key:<28} {t:18.6f} {r:18.6f} {t - r:14.6e}")
    print("-" * 80)
    t_tot = of_e.value_in_unit(EU)
    r_tot = ic_e.value_in_unit(EU)
    print(f"{'TOTAL':<28} {t_tot:18.6f} {r_tot:18.6f} {t_tot - r_tot:14.6e}")

    of_f_arr = of_f.value_in_unit(FU)
    ic_f_arr = ic_f_permuted.value_in_unit(FU)
    scale = (of_f_arr * of_f_arr).sum() + (ic_f_arr * ic_f_arr).sum()
    rel_dev = np.sqrt(((of_f_arr - ic_f_arr) ** 2).sum() / scale) if scale > 0 else 0.0
    print(f"\nForce deviation: {rel_dev:.2e}")

    n_vs_of = sum(of_sys.isVirtualSite(i) for i in range(of_sys.getNumParticles()))
    n_vs_ic = sum(ic_sys.isVirtualSite(i) for i in range(ic_sys.getNumParticles()))

    ok = True
    if de > 1e-2:
        print(f"FAIL: energy delta {de:.2e} > 1e-2 kcal/mol")
        ok = False
    if c_of != c_ic:
        print(f"FAIL: constraints {c_of} vs {c_ic}")
        ok = False
    if n_vs_of != n_vs_ic:
        print(f"FAIL: vsites {n_vs_of} vs {n_vs_ic}")
        ok = False
    if rel_dev > 1e-5:
        print(f"FAIL: force deviation {rel_dev:.2e} > 1e-5")
        ok = False
    if ok:
        print(f"PASS: {name} (dE={de:.1e}, constraints={c_of}, vsites={n_vs_of}, "
              f"atoms={len(perm)}, force_dev={rel_dev:.1e})")
    else:
        raise AssertionError(f"FAIL: {name}")


def add_atom(mol, an, fc, arom, name, res_name, res_num, chain="A", stereo=None):
    """Convenience wrapper for Molecule.add_atom with metadata."""
    return mol.add_atom(
        an, fc, arom,
        stereochemistry=stereo,
        name=name,
        metadata={"residue_name": res_name, "residue_number": str(res_num), "chain_id": chain},
    )

## Test 1: 2 External Bonds

Propane split into 3 residues — middle CH2 residue has 2 external bonds.

In [3]:
mol = Molecule()
# Residue A: CH3
c0 = add_atom(mol, 6, 0, False, "C1", "METHA", 1)
h0 = add_atom(mol, 1, 0, False, "H1", "METHA", 1)
h1 = add_atom(mol, 1, 0, False, "H2", "METHA", 1)
h2 = add_atom(mol, 1, 0, False, "H3", "METHA", 1)
# Residue B: CH2 (2 external bonds)
c1 = add_atom(mol, 6, 0, False, "C2", "METHB", 2)
h3 = add_atom(mol, 1, 0, False, "H4", "METHB", 2)
h4 = add_atom(mol, 1, 0, False, "H5", "METHB", 2)
# Residue C: CH3
c2 = add_atom(mol, 6, 0, False, "C3", "METHC", 3)
h5 = add_atom(mol, 1, 0, False, "H6", "METHC", 3)
h6 = add_atom(mol, 1, 0, False, "H7", "METHC", 3)
h7 = add_atom(mol, 1, 0, False, "H8", "METHC", 3)

mol.add_bond(c0, h0, 1, False)
mol.add_bond(c0, h1, 1, False)
mol.add_bond(c0, h2, 1, False)
mol.add_bond(c0, c1, 1, False)  # cross-residue
mol.add_bond(c1, h3, 1, False)
mol.add_bond(c1, h4, 1, False)
mol.add_bond(c1, c2, 1, False)  # cross-residue
mol.add_bond(c2, h5, 1, False)
mol.add_bond(c2, h6, 1, False)
mol.add_bond(c2, h7, 1, False)

mol.generate_conformers(n_conformers=1)
compare("propane (2 ext bonds)", mol, preload=True)

/Users/jeffreywagner/micromamba/envs/ommffs-dev/lib/python3.13/site-packages/openff/interchange/components/interchange.py:1119: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, functools._lru_cache_wrapper) and obj.__module__.startswith("openff.interchange"):


/Users/jeffreywagner/micromamba/envs/ommffs-dev/lib/python3.13/site-packages/openff/interchange/components/interchange.py:1119: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, functools._lru_cache_wrapper) and obj.__module__.startswith("openff.interchange"):
/Users/jeffreywagner/micromamba/envs/ommffs-dev/lib/python3.13/site-packages/openff/interchange/smirnoff/_create.py:151: PresetChargesAndVirtualSitesWarning: Preset charges were provided (via `charge_from_molecules`) alongside a force field that includes virtual site parameters. Note that virtual sites will be applied charges from the force field and cannot be given preset charges. Virtual sites may also affect the charges of their orientation atoms, even if those atoms are given preset charges.
  warnings.warn(



Component                                OMMFFS        Interchange          Delta
--------------------------------------------------------------------------------
CMMotionRemover                        0.000000           0.000000   0.000000e+00
HarmonicAngleForce                     1.825945           1.825945   0.000000e+00
HarmonicBondForce                      0.262207           0.262207   0.000000e+00
NonbondedForce                         1.409417           1.409417   0.000000e+00
PeriodicTorsionForce                   3.451863           3.451863   0.000000e+00
--------------------------------------------------------------------------------
TOTAL                                  6.949433           6.949433   1.776357e-15

Force deviation: 5.01e-17
PASS: propane (2 ext bonds) (dE=1.7e-15, constraints=8, vsites=0, atoms=11, force_dev=5.0e-17)


## Test 2: 3 External Bonds

Isobutane with central CH in its own residue, bonded to 3 CH3 residues.

In [4]:
mol = Molecule()
# Central residue: CH
c0 = add_atom(mol, 6, 0, False, "C0", "HUB", 1)
h0 = add_atom(mol, 1, 0, False, "H0", "HUB", 1)
# Arm 1: CH3
c1 = add_atom(mol, 6, 0, False, "C1", "ARM1", 2)
h1 = add_atom(mol, 1, 0, False, "H1", "ARM1", 2)
h2 = add_atom(mol, 1, 0, False, "H2", "ARM1", 2)
h3 = add_atom(mol, 1, 0, False, "H3", "ARM1", 2)
# Arm 2: CH3
c2 = add_atom(mol, 6, 0, False, "C2", "ARM2", 3)
h4 = add_atom(mol, 1, 0, False, "H4", "ARM2", 3)
h5 = add_atom(mol, 1, 0, False, "H5", "ARM2", 3)
h6 = add_atom(mol, 1, 0, False, "H6", "ARM2", 3)
# Arm 3: CH3
c3 = add_atom(mol, 6, 0, False, "C3", "ARM3", 4)
h7 = add_atom(mol, 1, 0, False, "H7", "ARM3", 4)
h8 = add_atom(mol, 1, 0, False, "H8", "ARM3", 4)
h9 = add_atom(mol, 1, 0, False, "H9", "ARM3", 4)

mol.add_bond(c0, h0, 1, False)
mol.add_bond(c0, c1, 1, False)  # cross-residue
mol.add_bond(c0, c2, 1, False)  # cross-residue
mol.add_bond(c0, c3, 1, False)  # cross-residue
mol.add_bond(c1, h1, 1, False)
mol.add_bond(c1, h2, 1, False)
mol.add_bond(c1, h3, 1, False)
mol.add_bond(c2, h4, 1, False)
mol.add_bond(c2, h5, 1, False)
mol.add_bond(c2, h6, 1, False)
mol.add_bond(c3, h7, 1, False)
mol.add_bond(c3, h8, 1, False)
mol.add_bond(c3, h9, 1, False)

mol.generate_conformers(n_conformers=1)
compare("isobutane (3 ext bonds)", mol, preload=True)

/Users/jeffreywagner/micromamba/envs/ommffs-dev/lib/python3.13/site-packages/openff/interchange/components/interchange.py:1119: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, functools._lru_cache_wrapper) and obj.__module__.startswith("openff.interchange"):


/Users/jeffreywagner/micromamba/envs/ommffs-dev/lib/python3.13/site-packages/openff/interchange/smirnoff/_create.py:151: PresetChargesAndVirtualSitesWarning: Preset charges were provided (via `charge_from_molecules`) alongside a force field that includes virtual site parameters. Note that virtual sites will be applied charges from the force field and cannot be given preset charges. Virtual sites may also affect the charges of their orientation atoms, even if those atoms are given preset charges.
  warnings.warn(



Component                                OMMFFS        Interchange          Delta
--------------------------------------------------------------------------------
CMMotionRemover                        0.000000           0.000000   0.000000e+00
HarmonicAngleForce                     3.606620           3.606620   0.000000e+00
HarmonicBondForce                      0.437725           0.437725   0.000000e+00
NonbondedForce                         3.356644           3.356644   0.000000e+00
PeriodicTorsionForce                   3.131146           3.131146   0.000000e+00
--------------------------------------------------------------------------------
TOTAL                                 10.532135          10.532135   0.000000e+00

Force deviation: 1.30e-16
PASS: isobutane (3 ext bonds) (dE=0.0e+00, constraints=10, vsites=0, atoms=14, force_dev=1.3e-16)


## Test 3: 4 External Bonds

Neopentane with central C as a single-atom residue, bonded to 4 CH3 residues.

In [5]:
mol = Molecule()
c0 = add_atom(mol, 6, 0, False, "C0", "CORE", 1)
arms = []
for i, rname in enumerate(["ARM1", "ARM2", "ARM3", "ARM4"], 2):
    ci = add_atom(mol, 6, 0, False, f"C{i-1}", rname, i)
    hi = []
    for j in range(3):
        hi.append(add_atom(mol, 1, 0, False, f"H{(i-2)*3+j}", rname, i))
    arms.append((ci, hi))

for ci, hi in arms:
    mol.add_bond(c0, ci, 1, False)  # cross-residue
    for h in hi:
        mol.add_bond(ci, h, 1, False)

mol.generate_conformers(n_conformers=1)
compare("neopentane (4 ext bonds)", mol, preload=True)


Component                                OMMFFS        Interchange          Delta
--------------------------------------------------------------------------------
CMMotionRemover                        0.000000           0.000000   0.000000e+00
HarmonicAngleForce                     4.996145           4.996145   0.000000e+00
HarmonicBondForce                      0.594440           0.594440   0.000000e+00
NonbondedForce                         2.155407           2.155407   0.000000e+00
PeriodicTorsionForce                   1.064224           1.064224   0.000000e+00
--------------------------------------------------------------------------------
TOTAL                                  8.810216           8.810216   0.000000e+00

Force deviation: 1.05e-16
PASS: neopentane (4 ext bonds) (dE=0.0e+00, constraints=12, vsites=0, atoms=17, force_dev=1.1e-16)


## Test 4: Virtual Site Spanning 3 Residues

N-methylformamide (CH3-NH-CHO) split into 3 residues with a custom
TrivalentLonePair on N. The vsite's orientation atoms (C-sp3, H, C-sp2)
span all 3 residues.

In [6]:
# Create force field with custom vsite parameter
off_ff_vs = OFFForceField(FF)
off_ff_vs.get_parameter_handler("VirtualSites")
off_ff_vs["VirtualSites"].add_parameter(
    parameter=VirtualSiteType(
        smirks="[#6X4:2][#7X3:1]([#1:3])[#6X3:4]",
        type="TrivalentLonePair",
        match="once",
        distance="0.5 * angstrom ** 1",
        inPlaneAngle="None",
        epsilon="1.0 * kilocalories_per_mole",
        sigma="1.5 * angstrom",
        charge_increment1="0.05 * elementary_charge ** 1",
        charge_increment2="-0.08 * elementary_charge ** 1",
        charge_increment3="-0.06 * elementary_charge ** 1",
        charge_increment4="-0.03 * elementary_charge ** 1",
    ),
)

mol = Molecule()
# Residue A: CH3
c0 = add_atom(mol, 6, 0, False, "C1", "METHY", 1)
h0 = add_atom(mol, 1, 0, False, "H1", "METHY", 1)
h1 = add_atom(mol, 1, 0, False, "H2", "METHY", 1)
h2 = add_atom(mol, 1, 0, False, "H3", "METHY", 1)
# Residue B: NH
n0 = add_atom(mol, 7, 0, False, "N1", "AMINO", 2)
h3 = add_atom(mol, 1, 0, False, "H4", "AMINO", 2)
# Residue C: CHO
c1 = add_atom(mol, 6, 0, False, "C2", "FORML", 3)
o0 = add_atom(mol, 8, 0, False, "O1", "FORML", 3)
h4 = add_atom(mol, 1, 0, False, "H5", "FORML", 3)

mol.add_bond(c0, h0, 1, False)
mol.add_bond(c0, h1, 1, False)
mol.add_bond(c0, h2, 1, False)
mol.add_bond(c0, n0, 1, False)  # cross-residue A-B
mol.add_bond(n0, h3, 1, False)
mol.add_bond(n0, c1, 1, False)  # cross-residue B-C
mol.add_bond(c1, o0, 2, False)
mol.add_bond(c1, h4, 1, False)

mol.generate_conformers(n_conformers=1)

print("Vsite SMARTS [#6X4:2][#7X3:1]([#1:3])[#6X3:4] matches:")
print("  :1=N (res AMINO), :2=C0 (res METHY), :3=H (res AMINO), :4=C1 (res FORML)")
print("  -> parent atoms span 3 residues")

compare("NMF vsite 3-residue", mol, off_ff=off_ff_vs, preload=True)

Vsite SMARTS [#6X4:2][#7X3:1]([#1:3])[#6X3:4] matches:
  :1=N (res AMINO), :2=C0 (res METHY), :3=H (res AMINO), :4=C1 (res FORML)
  -> parent atoms span 3 residues



Component                                OMMFFS        Interchange          Delta
--------------------------------------------------------------------------------
CMMotionRemover                        0.000000           0.000000   0.000000e+00
HarmonicAngleForce                     1.703174           1.703174   0.000000e+00
HarmonicBondForce                      0.638396           0.638396   0.000000e+00
NonbondedForce                        -8.514809          -8.514809   0.000000e+00
PeriodicTorsionForce                   2.668230           2.668230   0.000000e+00
--------------------------------------------------------------------------------
TOTAL                                 -3.505009          -3.505009   0.000000e+00

Force deviation: 1.74e-16
PASS: NMF vsite 3-residue (dE=0.0e+00, constraints=5, vsites=1, atoms=10, force_dev=1.7e-16)


## Test 5: Noncontiguous Residues with Identical Metadata

Propane where atoms are ordered so that the first and third residue groups
share the same `residue_name` and `residue_number`, but are separated by
a different residue. OpenMM should create 3 separate residues.

In [7]:
mol = Molecule()
# First group: CH3 -- residue "GLY" #1
c0 = add_atom(mol, 6, 0, False, "C1", "GLY", 1)
h0 = add_atom(mol, 1, 0, False, "H1", "GLY", 1)
h1 = add_atom(mol, 1, 0, False, "H2", "GLY", 1)
h2 = add_atom(mol, 1, 0, False, "H3", "GLY", 1)
# Middle group: CH2 -- residue "ALA" #2
c1 = add_atom(mol, 6, 0, False, "C2", "ALA", 2)
h3 = add_atom(mol, 1, 0, False, "H4", "ALA", 2)
h4 = add_atom(mol, 1, 0, False, "H5", "ALA", 2)
# Last group: CH3 -- SAME metadata as first: "GLY" #1
c2 = add_atom(mol, 6, 0, False, "C3", "GLY", 1)
h5 = add_atom(mol, 1, 0, False, "H6", "GLY", 1)
h6 = add_atom(mol, 1, 0, False, "H7", "GLY", 1)
h7 = add_atom(mol, 1, 0, False, "H8", "GLY", 1)

mol.add_bond(c0, h0, 1, False)
mol.add_bond(c0, h1, 1, False)
mol.add_bond(c0, h2, 1, False)
mol.add_bond(c0, c1, 1, False)
mol.add_bond(c1, h3, 1, False)
mol.add_bond(c1, h4, 1, False)
mol.add_bond(c1, c2, 1, False)
mol.add_bond(c2, h5, 1, False)
mol.add_bond(c2, h6, 1, False)
mol.add_bond(c2, h7, 1, False)

mol.generate_conformers(n_conformers=1)

# Verify OpenMM creates 3 residues despite shared metadata
omm_top = mol.to_topology().to_openmm()
res_list = list(omm_top.residues())
print(f"OpenMM residues: {len(res_list)}")
for r in res_list:
    atoms = [a.name for a in r.atoms()]
    print(f"  {r.name} #{r.id} chain={r.chain.id}: {atoms}")
assert len(res_list) == 3, f"Expected 3 residues, got {len(res_list)}"
assert res_list[0].name == res_list[2].name, "First and third should share name"
assert res_list[0].id == res_list[2].id, "First and third should share number"

compare("noncontiguous identical metadata", mol, preload=True)

OpenMM residues: 3
  GLY #1 chain=A: ['C1', 'H1', 'H2', 'H3']
  ALA #2 chain=A: ['C2', 'H4', 'H5']
  GLY #1 chain=A: ['C3', 'H6', 'H7', 'H8']


/Users/jeffreywagner/micromamba/envs/ommffs-dev/lib/python3.13/site-packages/openff/interchange/components/interchange.py:1119: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, functools._lru_cache_wrapper) and obj.__module__.startswith("openff.interchange"):
/Users/jeffreywagner/micromamba/envs/ommffs-dev/lib/python3.13/site-packages/openff/interchange/smirnoff/_create.py:151: PresetChargesAndVirtualSitesWarning: Preset charges were provided (via `charge_from_molecules`) alongside a force field that includes virtual site parameters. Note that virtual sites will be applied charges from the force field and cannot be given preset charges. Virtual sites may also affect the charges of their orientation atoms, even if those atoms are given preset charges.
  warnings.warn(



Component                                OMMFFS        Interchange          Delta
--------------------------------------------------------------------------------
CMMotionRemover                        0.000000           0.000000   0.000000e+00
HarmonicAngleForce                     1.825945           1.825945   0.000000e+00
HarmonicBondForce                      0.262207           0.262207   0.000000e+00
NonbondedForce                         1.409417           1.409417   0.000000e+00
PeriodicTorsionForce                   3.451863           3.451863   0.000000e+00
--------------------------------------------------------------------------------
TOTAL                                  6.949433           6.949433   1.776357e-15

Force deviation: 5.01e-17
PASS: noncontiguous identical metadata (dE=1.7e-15, constraints=8, vsites=0, atoms=11, force_dev=5.0e-17)


## Test 6: Molecule Spanning Multiple Chains

Ethane with one CH3 in chain A, the other in chain B. Tests that
`_preprocess_residue` correctly finds cross-chain bonds.

In [8]:
mol = Molecule()
# Chain A
c0 = add_atom(mol, 6, 0, False, "C1", "ETHA", 1, chain="A")
h0 = add_atom(mol, 1, 0, False, "H1", "ETHA", 1, chain="A")
h1 = add_atom(mol, 1, 0, False, "H2", "ETHA", 1, chain="A")
h2 = add_atom(mol, 1, 0, False, "H3", "ETHA", 1, chain="A")
# Chain B
c1 = add_atom(mol, 6, 0, False, "C2", "ETHB", 1, chain="B")
h3 = add_atom(mol, 1, 0, False, "H4", "ETHB", 1, chain="B")
h4 = add_atom(mol, 1, 0, False, "H5", "ETHB", 1, chain="B")
h5 = add_atom(mol, 1, 0, False, "H6", "ETHB", 1, chain="B")

mol.add_bond(c0, h0, 1, False)
mol.add_bond(c0, h1, 1, False)
mol.add_bond(c0, h2, 1, False)
mol.add_bond(c0, c1, 1, False)  # cross-chain bond!
mol.add_bond(c1, h3, 1, False)
mol.add_bond(c1, h4, 1, False)
mol.add_bond(c1, h5, 1, False)

mol.generate_conformers(n_conformers=1)

omm_top = mol.to_topology().to_openmm()
chains = list(omm_top.chains())
print(f"OpenMM chains: {len(chains)}")
for ch in chains:
    for r in ch.residues():
        atoms = [a.name for a in r.atoms()]
        print(f"  Chain {ch.id}, {r.name} #{r.id}: {atoms}")
assert len(chains) == 2, f"Expected 2 chains, got {len(chains)}"

compare("ethane cross-chain", mol, preload=True)

OpenMM chains: 2
  Chain A, ETHA #1: ['C1', 'H1', 'H2', 'H3']
  Chain B, ETHB #1: ['C2', 'H4', 'H5', 'H6']


/Users/jeffreywagner/micromamba/envs/ommffs-dev/lib/python3.13/site-packages/openff/interchange/components/interchange.py:1119: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, functools._lru_cache_wrapper) and obj.__module__.startswith("openff.interchange"):


/Users/jeffreywagner/micromamba/envs/ommffs-dev/lib/python3.13/site-packages/openff/interchange/components/interchange.py:1119: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, functools._lru_cache_wrapper) and obj.__module__.startswith("openff.interchange"):
/Users/jeffreywagner/micromamba/envs/ommffs-dev/lib/python3.13/site-packages/openff/interchange/smirnoff/_create.py:151: PresetChargesAndVirtualSitesWarning: Preset charges were provided (via `charge_from_molecules`) alongside a force field that includes virtual site parameters. Note that virtual sites will be applied charges from the force field and cannot be given preset charges. Virtual sites may also affect the charges of their orientation atoms, even if those atoms are given preset charges.
  warnings.warn(



Component                                OMMFFS        Interchange          Delta
--------------------------------------------------------------------------------
CMMotionRemover                        0.000000           0.000000   0.000000e+00
HarmonicAngleForce                     2.036270           2.036270   0.000000e+00
HarmonicBondForce                      0.287720           0.287720   0.000000e+00
NonbondedForce                         1.064052           1.064052   0.000000e+00
PeriodicTorsionForce                   3.098529           3.098529   0.000000e+00
--------------------------------------------------------------------------------
TOTAL                                  6.486571           6.486571   0.000000e+00

Force deviation: 8.94e-17
PASS: ethane cross-chain (dE=0.0e+00, constraints=6, vsites=0, atoms=8, force_dev=8.9e-17)


## Test 7: Stereochemical Isomers

(R)- and (S)-2-butanol split into 2 residues each. Tests:
- Each isomer individually
- Both in the same topology (registering only one template)

**Finding:** Registering both stereoisomers and preloading their templates
causes OpenMM to raise `Multiple non-identical matching templates found`
because template matching ignores stereochemistry, so both templates match
both residues. The workaround is to register only one stereoisomer.

In [9]:
def build_2_butanol(stereo, res_split=True):
    """Build 2-butanol: CH3-C*H(OH)-CH2-CH3 with specified stereochemistry."""
    mol = Molecule()
    c0 = add_atom(mol, 6, 0, False, "C1", "BUT1", 1)
    h0 = add_atom(mol, 1, 0, False, "H1", "BUT1", 1)
    h1 = add_atom(mol, 1, 0, False, "H2", "BUT1", 1)
    h2 = add_atom(mol, 1, 0, False, "H3", "BUT1", 1)
    c1 = add_atom(mol, 6, 0, False, "C2", "BUT1", 1, stereo=stereo)
    h3 = add_atom(mol, 1, 0, False, "H4", "BUT1", 1)
    o0 = add_atom(mol, 8, 0, False, "O1", "BUT1", 1)
    ho = add_atom(mol, 1, 0, False, "HO", "BUT1", 1)
    rn2 = "BUT2" if res_split else "BUT1"
    rnum2 = 2 if res_split else 1
    c2 = add_atom(mol, 6, 0, False, "C3", rn2, rnum2)
    h4 = add_atom(mol, 1, 0, False, "H5", rn2, rnum2)
    h5 = add_atom(mol, 1, 0, False, "H6", rn2, rnum2)
    c3 = add_atom(mol, 6, 0, False, "C4", rn2, rnum2)
    h6 = add_atom(mol, 1, 0, False, "H7", rn2, rnum2)
    h7 = add_atom(mol, 1, 0, False, "H8", rn2, rnum2)
    h8 = add_atom(mol, 1, 0, False, "H9", rn2, rnum2)

    mol.add_bond(c0, h0, 1, False)
    mol.add_bond(c0, h1, 1, False)
    mol.add_bond(c0, h2, 1, False)
    mol.add_bond(c0, c1, 1, False)
    mol.add_bond(c1, h3, 1, False)
    mol.add_bond(c1, o0, 1, False)
    mol.add_bond(o0, ho, 1, False)
    mol.add_bond(c1, c2, 1, False)  # cross-residue if split
    mol.add_bond(c2, h4, 1, False)
    mol.add_bond(c2, h5, 1, False)
    mol.add_bond(c2, c3, 1, False)
    mol.add_bond(c3, h6, 1, False)
    mol.add_bond(c3, h7, 1, False)
    mol.add_bond(c3, h8, 1, False)

    mol.generate_conformers(n_conformers=1)
    return mol


mol_r = build_2_butanol("R")
mol_s = build_2_butanol("S")

print(f"(R)-2-butanol SMILES: {mol_r.to_smiles()}")
print(f"(S)-2-butanol SMILES: {mol_s.to_smiles()}")
print(f"Same canonical SMILES? {mol_r.to_smiles() == mol_s.to_smiles()}")

# Test each stereoisomer separately
compare("(R)-2-butanol", mol_r, preload=True)
compare("(S)-2-butanol", mol_s, preload=True)

# Test with both in same topology -- register only one stereoisomer.
# Registering both causes "Multiple non-identical matching templates found"
# because template matching ignores stereochemistry.
print("\nCombined test: register only (R), parameterize both R and S")
compare("R+S 2-butanol (single template)", [mol_r, mol_s], preload=True,
        register_mols=[mol_r])

(R)-2-butanol SMILES: [H][O][C@]([H])([C]([H])([H])[H])[C]([H])([H])[C]([H])([H])[H]
(S)-2-butanol SMILES: [H][O][C@@]([H])([C]([H])([H])[H])[C]([H])([H])[C]([H])([H])[H]
Same canonical SMILES? False


/Users/jeffreywagner/micromamba/envs/ommffs-dev/lib/python3.13/site-packages/openff/interchange/components/interchange.py:1119: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, functools._lru_cache_wrapper) and obj.__module__.startswith("openff.interchange"):


/Users/jeffreywagner/micromamba/envs/ommffs-dev/lib/python3.13/site-packages/openff/interchange/smirnoff/_create.py:151: PresetChargesAndVirtualSitesWarning: Preset charges were provided (via `charge_from_molecules`) alongside a force field that includes virtual site parameters. Note that virtual sites will be applied charges from the force field and cannot be given preset charges. Virtual sites may also affect the charges of their orientation atoms, even if those atoms are given preset charges.
  warnings.warn(



Component                                OMMFFS        Interchange          Delta
--------------------------------------------------------------------------------
CMMotionRemover                        0.000000           0.000000   0.000000e+00
HarmonicAngleForce                     1.997063           1.997063   0.000000e+00
HarmonicBondForce                      1.162495           1.162495   0.000000e+00
NonbondedForce                        -6.770656          -6.770656   0.000000e+00
PeriodicTorsionForce                  11.148743          11.148743   0.000000e+00
--------------------------------------------------------------------------------
TOTAL                                  7.537645           7.537645   0.000000e+00

Force deviation: 7.54e-17
PASS: (R)-2-butanol (dE=0.0e+00, constraints=10, vsites=0, atoms=15, force_dev=7.5e-17)



Component                                OMMFFS        Interchange          Delta
--------------------------------------------------------------------------------
CMMotionRemover                        0.000000           0.000000   0.000000e+00
HarmonicAngleForce                     3.375900           3.375900   0.000000e+00
HarmonicBondForce                      0.959415           0.959415   0.000000e+00
NonbondedForce                        -6.327585          -6.327585   0.000000e+00
PeriodicTorsionForce                   5.730728           5.730728   0.000000e+00
--------------------------------------------------------------------------------
TOTAL                                  3.738458           3.738458   4.440892e-16

Force deviation: 6.02e-17
PASS: (S)-2-butanol (dE=4.2e-16, constraints=10, vsites=0, atoms=15, force_dev=6.0e-17)

Combined test: register only (R), parameterize both R and S



Component                                OMMFFS        Interchange          Delta
--------------------------------------------------------------------------------
CMMotionRemover                        0.000000           0.000000   0.000000e+00
HarmonicAngleForce                     5.372963           5.372963   0.000000e+00
HarmonicBondForce                      2.121910           2.121910   0.000000e+00
NonbondedForce               1155990609357408000.000000 1155990609357408000.000000   0.000000e+00
PeriodicTorsionForce                  16.879471          16.879471   0.000000e+00
--------------------------------------------------------------------------------
TOTAL                        1155990609357408000.000000 1155990609357408000.000000   0.000000e+00

Force deviation: 0.00e+00
PASS: R+S 2-butanol (single template) (dE=0.0e+00, constraints=20, vsites=0, atoms=30, force_dev=0.0e+00)


In [10]:
print("All multi-residue topology tests passed!")

All multi-residue topology tests passed!
